In [1]:
import sqlite3
import pandas as pd
import numpy as np

# Load clean data — no cleaning steps needed
conn = sqlite3.connect('florida_leads.db')
df = pd.read_sql_query("SELECT * FROM leads_clean", conn)
conn.close()

print(f"Records loaded   : {len(df):,}")
print(f"Columns          : {len(df.columns)}")
print(f"never_sold count : {df['never_sold'].sum():,}")
print(f"Data types OK    : {df['ACT_YR_BLT'].dtype}")

Records loaded   : 4,837
Columns          : 26
never_sold count : 4,368
Data types OK    : int64


In [2]:
# Feature 1 — Property Age
# How old is the building?
# Older property + long ownership = stronger motivated seller signal

df['property_age'] = 2025 - df['ACT_YR_BLT']

print(f"property_age created successfully")
print(f"Min age  : {df['property_age'].min()} years")
print(f"Max age  : {df['property_age'].max()} years")
print(f"Mean age : {df['property_age'].mean():.1f} years")
print(f"Negative values (error check): {(df['property_age'] < 0).sum()}")

property_age created successfully
Min age  : 1 years
Max age  : 130 years
Mean age : 36.6 years
Negative values (error check): 0


In [3]:
# Feature 2 — Is Corporate Owner
CORPORATE_KEYWORDS = [
    'LLC', 'INC', 'CORP', 'TRUST', 'ESTATE',
    'PROPERTIES', 'GROUP', 'HOLDINGS', 'PARTNERS',
    'INVESTMENTS', 'REALTY', 'ENTERPRISES', 'LTD'
]

df['is_corporate_owner'] = (
    df['OWN_NAME']
    .astype(str)
    .str.upper()
    .str.contains('|'.join(CORPORATE_KEYWORDS), na=False)
    .astype(int)
)

print(f"Individual owners  : {(df['is_corporate_owner'] == 0).sum():,}")
print(f"Corporate owners   : {(df['is_corporate_owner'] == 1).sum():,}")
print(f"Corporate %        : {df['is_corporate_owner'].mean() * 100:.1f}%")
print()
print("Sample corporate names found:")
print(df[df['is_corporate_owner'] == 1]['OWN_NAME'].head(6).to_string())

Individual owners  : 4,638
Corporate owners   : 199
Corporate %        : 4.1%

Sample corporate names found:
2                    BLC FL LLC
7     NSA PROPERTY HOLDINGS LLC
8               FLINCHUM MARC C
15           BRENT C SALLEY LLC
16           BRENT C SALLEY LLC
39      YEAGO PROPERTIES II LLC


In [4]:
# is absente owner !
df['is_absentee_owner'] = (df['JV_HMSTD'] == 0).astype(int)

print(f"Owner-occupied     : {(df['is_absentee_owner'] == 0).sum():,}")
print(f"Absentee owners    : {(df['is_absentee_owner'] == 1).sum():,}")
print(f"Absentee %         : {df['is_absentee_owner'].mean() * 100:.1f}%")

Owner-occupied     : 3,375
Absentee owners    : 1,462
Absentee %         : 30.2%


In [5]:
#is out of state owner ?
df['is_out_of_state_owner'] = (
    (df['OWN_STATE_'].astype(str).str.strip() != 'FL') &
    (df['OWN_STATE_'].astype(str).str.strip() != '')
).astype(int)

print(f"Florida owners     : {(df['is_out_of_state_owner'] == 0).sum():,}")
print(f"Out-of-state owners: {(df['is_out_of_state_owner'] == 1).sum():,}")
print(f"Out-of-state %     : {df['is_out_of_state_owner'].mean() * 100:.1f}%")
print()
print("Top states of out-of-state owners:")
print(df[df['is_out_of_state_owner'] == 1]['OWN_STATE_'].value_counts().head(8))

Florida owners     : 4,678
Out-of-state owners: 159
Out-of-state %     : 3.3%

Top states of out-of-state owners:
OWN_STATE_
GA    20
CA    13
TX    13
IN     9
NC     9
NY     7
CO     6
VA     6
Name: count, dtype: int64


In [6]:
#land ratio
df['land_ratio'] = df['LND_VAL'] / df['JV'].replace(0, 1)

print(f"Min land ratio  : {df['land_ratio'].min():.3f}")
print(f"Max land ratio  : {df['land_ratio'].max():.3f}")
print(f"Mean land ratio : {df['land_ratio'].mean():.3f}")
print(f"Values above 1.0: {(df['land_ratio'] > 1.0).sum()} (data errors if any)")

Min land ratio  : 0.011
Max land ratio  : 1.000
Mean land ratio : 0.314
Values above 1.0: 0 (data errors if any)


In [7]:
#price sqft!
df['price_per_sqft'] = df['JV'] / df['TOT_LVG_AR']

print(f"Valid calculations : {df['price_per_sqft'].notna().sum():,}")
print(f"NaN (vacant land)  : {df['price_per_sqft'].isna().sum():,}")
print(f"Min  : ${df['price_per_sqft'].min():.2f}")
print(f"Max  : ${df['price_per_sqft'].max():.2f}")
print(f"Mean : ${df['price_per_sqft'].mean():.2f}")

Valid calculations : 4,594
NaN (vacant land)  : 243
Min  : $3.64
Max  : $648.34
Mean : $124.86


In [8]:
#lot coverage ratio 
df['lot_coverage_ratio'] = df['TOT_LVG_AR'] / df['LND_SQFOOT'].replace(0, np.nan)

print(f"Valid calculations : {df['lot_coverage_ratio'].notna().sum():,}")
print(f"NaN records        : {df['lot_coverage_ratio'].isna().sum():,}")
print(f"Min  : {df['lot_coverage_ratio'].min():.4f}")
print(f"Max  : {df['lot_coverage_ratio'].max():.4f}")
print(f"Mean : {df['lot_coverage_ratio'].mean():.4f}")
print(f"Values above 1.0   : {(df['lot_coverage_ratio'] > 1.0).sum()} (check these)")

Valid calculations : 4,594
NaN records        : 243
Min  : 0.0005
Max  : 1.6073
Mean : 0.1028
Values above 1.0   : 23 (check these)


In [9]:
#renovation gap 
df['renovation_gap'] = df['EFF_YR_BLT'] - df['ACT_YR_BLT']

print(f"Min gap  : {df['renovation_gap'].min()}")
print(f"Max gap  : {df['renovation_gap'].max()}")
print(f"Mean gap : {df['renovation_gap'].mean():.1f}")
print(f"Zero gap (never renovated): {(df['renovation_gap'] == 0).sum():,}")
print(f"Zero gap %                : {(df['renovation_gap'] == 0).mean()*100:.1f}%")
print()
print("Properties with renovation gap > 20 years:")
print(df[df['renovation_gap'] > 20][['ACT_YR_BLT','EFF_YR_BLT','renovation_gap']].head(5))

Min gap  : -74
Max gap  : 116
Mean gap : 11.2
Zero gap (never renovated): 2,089
Zero gap %                : 43.2%

Properties with renovation gap > 20 years:
    ACT_YR_BLT  EFF_YR_BLT  renovation_gap
5         1940        1995              55
17        1989        2010              21
37        1962        1995              33
61        1948        1995              47
62        1964        1995              31


In [10]:
#value tier
tier_bins   = [0, 50000, 150000, 300000, 500000, float('inf')]
tier_labels = ['distressed', 'entry', 'mid', 'upper', 'premium']

df['value_tier'] = pd.cut(
    df['JV'],
    bins=tier_bins,
    labels=tier_labels,
    right=True
)

print("Value tier distribution:")
print(df['value_tier'].value_counts().sort_index())
print()
print(f"Wholesale targets (distressed + entry): "
      f"{df['value_tier'].isin(['distressed','entry']).sum():,} records")

Value tier distribution:
value_tier
distressed     329
entry         1466
mid           2135
upper          875
premium         32
Name: count, dtype: int64

Wholesale targets (distressed + entry): 1,795 records


In [11]:
#nbrhood value ratio
nbrhd_mean_jv = df.groupby('NBRHD_CD')['JV'].transform('mean')
df['neighborhood_value_ratio'] = df['JV'] / nbrhd_mean_jv

print(f"Min ratio  : {df['neighborhood_value_ratio'].min():.3f}")
print(f"Max ratio  : {df['neighborhood_value_ratio'].max():.3f}")
print(f"Mean ratio : {df['neighborhood_value_ratio'].mean():.3f}")
print(f"Undervalued (ratio < 0.8) : {(df['neighborhood_value_ratio'] < 0.8).sum():,} records")
print(f"Overvalued  (ratio > 1.2) : {(df['neighborhood_value_ratio'] > 1.2).sum():,} records")
print()
print("Neighborhood mean JV sample (top 5 by record count):")
print(df.groupby('NBRHD_CD')['JV'].agg(['mean','count']).sort_values('count', ascending=False).head())

Min ratio  : 0.021
Max ratio  : 4.606
Mean ratio : 1.000
Undervalued (ratio < 0.8) : 1,429 records
Overvalued  (ratio > 1.2) : 1,112 records

Neighborhood mean JV sample (top 5 by record count):
                    mean  count
NBRHD_CD                       
323400.00  152701.126667    300
233328.06  251549.914286    245
233328.03  324619.798354    243
232300.00  181558.775000    160
322400.00  138628.343284    134


In [12]:
#distressed sale history
df['distressed_sale_history'] = (
    df['QUAL_CD1'].astype(str).str.strip().str.upper() == 'U'
).astype(int)

print(f"Clean sale history     : {(df['distressed_sale_history'] == 0).sum():,}")
print(f"Distressed sale history: {(df['distressed_sale_history'] == 1).sum():,}")
print(f"Distressed %           : {df['distressed_sale_history'].mean() * 100:.1f}%")
print()
print("QUAL_CD1 value breakdown:")
print(df['QUAL_CD1'].astype(str).str.strip().str.upper().value_counts())

Clean sale history     : 4,837
Distressed sale history: 0
Distressed %           : 0.0%

QUAL_CD1 value breakdown:
QUAL_CD1
      4302
11     311
01     144
02      43
05      18
37       8
30       4
16       3
18       1
03       1
21       1
12       1
Name: count, dtype: int64


In [13]:
# Count how many motivated signals each property has
signal_score = (
    df['never_sold'] +
    df['is_absentee_owner'] +
    df['is_out_of_state_owner'] +
    (df['property_age'] > 30).astype(int) +
    df['value_tier'].isin(['distressed', 'entry']).astype(int) +
    (df['neighborhood_value_ratio'] < 0.8).astype(int) +
    df['distressed_sale_history']
)

df['hot_lead'] = (signal_score >= 3).astype(int)

print(f"Total signals possible : 7")
print(f"Signal score distribution:")
print(signal_score.value_counts().sort_index())
print()
print(f"Not hot leads : {(df['hot_lead'] == 0).sum():,}")
print(f"Hot leads     : {(df['hot_lead'] == 1).sum():,}")
print(f"Hot lead %    : {df['hot_lead'].mean() * 100:.1f}%")

Total signals possible : 7
Signal score distribution:
0     114
1    1453
2    1263
3     816
4     655
5     489
6      47
Name: count, dtype: int64

Not hot leads : 2,830
Hot leads     : 2,007
Hot lead %    : 41.5%


In [14]:
conn = sqlite3.connect("florida_leads.db")
df.to_sql("leads_features", conn, if_exists="replace", index=False)
conn.close()
print(f"Saved {len(df):,} records to leads_features table")

Saved 4,837 records to leads_features table
